## 1. Setup & Data Loading {#setup--data-loading}

Import required libraries and load the feature-selected dataset and baseline models from previous steps.

In [ ]:
# Core
import os
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.base import clone
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# Bayesian optimization (scikit-optimize)
from skopt import BayesSearchCV
from skopt.space import Real, Integer

# Model
from xgboost import XGBClassifier


### 1.2 Load Processed Data

In [ ]:
# Define data directory
current_user = os.getenv('USER')
PROCESSED_DIR = Path(f"/home/{current_user}/local-share/processed")

# Load data and models
df = pd.read_csv(PROCESSED_DIR / 'feature_selection.csv')

# Load 'model_comparison_analysis.pkl'
model_data = joblib.load(PROCESSED_DIR / 'model_comparison_analysis.pkl')

### 1.3 Rebuild splits from set and define target

In [ ]:
# Purpose: Define the target column and reconstruct the train/validation/test splits
#          using the precomputed 'set' labels to avoid leakage and keep tuning consistent.

target_col = 'sdo5_degree_drop_out'

# Split data based on the 'set' column
train_df = df[df['set'] == 'train'].copy()
val_df   = df[df['set'] == 'val'].copy()
test_df  = df[df['set'] == 'test'].copy()

print(f"Training set shape: {train_df.shape}")
print(f"Validation set shape: {val_df.shape}")
print(f"Test set shape: {test_df.shape}")


### 1.4 Model Selection {#model-selection}

Based on the model comparison results, select the top performing models for hyperparameter tuning.

In [ ]:
# Purpose: Load saved model-comparison outputs and select only the previously chosen best model (XGBoost)
#          to continue with hyperparameter tuning in a controlled and reproducible way.

cv_results = model_data['cv_results']

best_model_name = model_data['best_model_name']
if best_model_name != "XGBoost":
    raise ValueError(f"Expected XGBoost as best model, but got: {best_model_name}")

models_to_tune = {"XGBoost": model_data['models']["XGBoost"]}

print("Model selected for hyperparameter tuning:")
print("- XGBoost")

## 2. Hyperparameter Search Space Configuration 

### 2.1 Define hyperparameter search space for XGB

In [ ]:
# Purpose: Define the Bayesian hyperparameter search space for XGBoost.
#          The search space is aligned with the baseline configuration used in the
#          model comparison notebook, preserving the same set of performance-relevant
#          hyperparameters (depth, learning rate, trees, sampling, and regularization).
#          Non-performance parameters (e.g. random_state, verbosity, eval_metric, n_jobs)
#          are intentionally kept fixed and excluded from tuning for reproducibility
#          and interpretability.

xgb_search_space = {
    'n_estimators': Integer(50, 400, name='n_estimators'),
    'max_depth': Integer(3, 8, name='max_depth'),
    'learning_rate': Real(0.01, 0.2, prior='log-uniform', name='learning_rate'),
    'subsample': Real(0.6, 1.0, prior='uniform', name='subsample'),
    'colsample_bytree': Real(0.6, 1.0, prior='uniform', name='colsample_bytree'),
    'reg_alpha': Real(1e-4, 1.0, prior='log-uniform', name='reg_alpha'),
    'reg_lambda': Real(0.5, 5.0, prior='log-uniform', name='reg_lambda'),
    'min_child_weight': Integer(1, 10, name='min_child_weight')
}

print("XGBoost hyperparameter search space defined.")

### 2.2 Build X_train_val, y_train_val, compute SPW, create tuned base estimator

In [ ]:
# Purpose: Reconstruct X_train_val and y_train_val from the existing train/val splits,
#          compute a fixed scale_pos_weight on train+val only, and create a fresh XGB estimator
#          that will be tuned with consistent imbalance handling, ie 
#          evaluating all hyperparameter configurations under the same class-imbalance assumption.

# Define feature columns (exclude split label + target)
exclude_cols = ['set', target_col]
feature_cols = [c for c in df.columns if c not in exclude_cols]

# Build train+val matrices for tuning
train_val_df = pd.concat([train_df, val_df], axis=0).reset_index(drop=True)

X_train_val = train_val_df[feature_cols].copy()
y_train_val = train_val_df[target_col].copy()

# Compute and fix scale_pos_weight using train+val only
neg = (y_train_val == 0).sum()
pos = (y_train_val == 1).sum()
spw = (neg / pos) if pos > 0 else 1.0

xgb_base = clone(models_to_tune["XGBoost"]).set_params(scale_pos_weight=spw)

print(f"Number of features: {len(feature_cols)}")
print(f"Train+Val shape: {X_train_val.shape}")
print(f"Fixed scale_pos_weight: {spw:.3f}")

## 3. Hyperparameter Optimization {#hyperparameter-optimization}

### 3.1 Define CV strategy and scoring (PR-AUC is the refit metric)

In [ ]:
# Purpose: Define a stable cross-validation strategy (RepeatedStratifiedKFold) and
#          a multi-metric scoring dictionary where PR-AUC is the primary optimization target.
#          Other metrics are tracked for reporting and sanity checks.

N_FOLDS = 10
N_REPEATS = 5

cv_strategy = RepeatedStratifiedKFold(
    n_splits=N_FOLDS,
    n_repeats=N_REPEATS,
    random_state=42
)

scoring = {
    'pr_auc': 'average_precision',
    'roc_auc': 'roc_auc',
    'accuracy': 'accuracy',
    'f1': 'f1',
    'recall': 'recall',
    'precision': 'precision'
}

print(f"CV strategy: {N_FOLDS}-fold × {N_REPEATS} repeats = {N_FOLDS * N_REPEATS} evaluations per trial")
print(f"Scoring metrics: {list(scoring.keys())}")
print("Refit metric: pr_auc")

### 3.2 Run Bayesian optimization (BayesSearchCV)

Why Bayesian optimization :

XGBoost has continuous and interacting hyperparameters; Bayesian optimization explores these far more efficiently than GridSearch or RandomSearch.

Each evaluation is expensive (repeated stratified CV); Bayesian optimization minimizes wasted trials.

PR-AUC is noisy and harder to optimize; Bayesian methods handle this better than brute-force searches.

 The goal was fine-tuning, not broad exploration—Bayesian optimization is ideal for local refinement.

It aligns with the existing careful CV, imbalance handling, and statistical evaluation setup.

In [ ]:
# Purpose: Run Bayesian hyperparameter optimization for XGBoost.
#          The search selects the best configuration by PR-AUC (Average Precision),
#          while storing all other metrics for later comparison and reporting.

N_ITER = 30  # number of times the hyperparameter space is explored, with 10x5 CV setup.

bayes_search = BayesSearchCV(
    estimator=xgb_base,            # already has fixed scale_pos_weight
    search_spaces=xgb_search_space,
    n_iter=N_ITER,
    scoring=scoring,
    refit='pr_auc',
    cv=cv_strategy,
    n_jobs=-1,
    random_state=42,
    verbose=0,
    return_train_score=True
)

bayes_search.fit(X_train_val, y_train_val)

print("Bayesian optimization complete.")
print(f"Best PR-AUC (CV mean): {bayes_search.best_score_:.4f}")
print("Best hyperparameters:")
print(bayes_search.best_params_)

#### Results: 

Baseline vs tuned performance:

Baseline PR-AUC (model comparison): 0.63

PR-AUC after Bayesian tuning: 0.64

#### Why this improvement matters:

The evaluation setup is strong (stratified CV with repeats).

PR-AUC is difficult to improve, especially in imbalanced settings.

The baseline XGBoost was already well configured.

#### What Bayesian optimization confirmed:

The tuning refined rather than overhauled the model.

Gains came from more trees and slower learning, not increased complexity.

Tree depth remained unchanged, supporting model stability.

#### Key takeaway:

Bayesian optimization polished an already strong XGBoost model, yielding a meaningful improvement in precision–recall performance for dropout prediction.

## 4. Generalization

### 4.1 Summarize tuned CV performance (best trial) across all metrics

In [ ]:
# Purpose: Extract and report the cross-validated performance of the selected best Bayesian trial.
# Justification: This provides transparency into what the optimizer actually optimized (PR-AUC),
# allows comparison between baseline CV, tuned CV, and final test performance, and helps verify
# that observed test-set behavior reflects generalization rather than overfitting to CV.


results = pd.DataFrame(bayes_search.cv_results_)
best_idx = bayes_search.best_index_

summary = {
    'PR_AUC_mean': results.loc[best_idx, 'mean_test_pr_auc'],
    'PR_AUC_std': results.loc[best_idx, 'std_test_pr_auc'],
    'ROC_AUC_mean': results.loc[best_idx, 'mean_test_roc_auc'],
    'ROC_AUC_std': results.loc[best_idx, 'std_test_roc_auc'],
    'Accuracy_mean': results.loc[best_idx, 'mean_test_accuracy'],
    'F1_mean': results.loc[best_idx, 'mean_test_f1'],
    'Recall_mean': results.loc[best_idx, 'mean_test_recall'],
    'Precision_mean': results.loc[best_idx, 'mean_test_precision']
}

print("\nBest tuned XGBoost CV performance (selected by PR-AUC):")
for k, v in summary.items():
    print(f"  {k}: {v:.4f}")


### 4.2 Train final tuned model on full train+val and evaluate on test (including PR-AUC)

In [ ]:
# Purpose: Fit the tuned XGBoost model once on the full Train+Validation set and evaluate it once on the
#          untouched Test set using the same threshold-independent metrics used in model comparison.
#          This final evaluation is necessary to confirm that the CV improvement generalizes to future/unseen cohorts.


# --- 1) Build Train+Val and Test matrices ---
train_val_df = pd.concat([train_df, val_df], axis=0).reset_index(drop=True)

X_train_val = train_val_df[feature_cols].copy()
y_train_val = train_val_df[target_col].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df[target_col].copy()

# --- 2) Compute scale_pos_weight on Train+Val (fixed imbalance handling) ---
neg = (y_train_val == 0).sum()
pos = (y_train_val == 1).sum()
spw = (neg / pos) if pos > 0 else 1.0

# --- 3) Rebuild final XGB model with best params + fixed settings ---
best_params = bayes_search.best_params_  # assumes BayesSearchCV already ran
final_xgb = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    verbosity=0,
    n_jobs=-1,
    scale_pos_weight=spw,
    **best_params
)

# --- 4) Fit on Train+Val ---
final_xgb.fit(X_train_val, y_train_val)

# --- 5) Predict probabilities on Test ---
y_test_proba = final_xgb.predict_proba(X_test)[:, 1]

# --- 6) Primary metrics (threshold-free; consistent with model comparison) ---
pr_auc = average_precision_score(y_test, y_test_proba)
roc_auc = roc_auc_score(y_test, y_test_proba)

print("\nFINAL TEST SET EVALUATION (consistent with model comparison):")
print(f"  PR-AUC (primary):   {pr_auc:.4f}")
print(f"  ROC-AUC (secondary): {roc_auc:.4f}")

The final test evaluation confirms that the tuned XGBoost model maintains strong precision–recall performance on an unseen cohort: 

PR-AUC improved from ≈0.632 (baseline CV) to 0.638 during Bayesian cross-validation, and achieved 0.623 on the held-out test set. 

While the full CV gain was not completely reproduced on the test data, performance remained close to the original baseline, 

indicating that the tuning improved robustness without inducing overfitting.

## 5. Save tuned model + tuning outputs

In [ ]:
# Purpose: Save the final tuned XGBoost model and the key tuning/evaluation outputs
#          (best hyperparameters, CV PR-AUC, test PR/ROC-AUC, and training metadata)
#          so results are reproducible and can be loaded in any notebook later.


# --- 1) Define save paths ---
current_user = os.getenv("USER")
PROCESSED_DIR = Path(f"/home/{current_user}/local-share/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

model_path = PROCESSED_DIR / "xgb_final_tuned_trainval.pkl"
outputs_path = PROCESSED_DIR / "xgb_tuning_outputs.pkl"

# --- 2) Collect tuning + evaluation outputs ---
tuning_outputs = {
    "timestamp": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
    "best_model_name": "XGBoost",
    "best_params": dict(bayes_search.best_params_),          # Bayesian best hyperparameters
    "best_cv_pr_auc": float(bayes_search.best_score_),       # CV mean PR-AUC from BayesSearchCV
    "final_scale_pos_weight": float(spw),                    # fixed imbalance weight used in final fit
    "test_metrics_threshold_free": {                         # final test evaluation (threshold-free)
        "pr_auc": float(pr_auc),
        "roc_auc": float(roc_auc),
    },
    "feature_cols": list(feature_cols),
    "target_col": target_col,
    "train_val_size": int(len(y_train_val)),
    "test_size": int(len(y_test)),
}

# --- 3) Save model + outputs ---
joblib.dump(final_xgb, model_path)
joblib.dump(tuning_outputs, outputs_path)

print("Saved tuned model to:", model_path)
print("Saved tuning outputs to:", outputs_path)
